# 🏒 Hockey Board Segmentation — Option B (Direct Inference)

This notebook implements the optimal strategy for **Direct Inference** (Option B).
Instead of warping frames at runtime, we train directly on the raw, angled broadcast frames.
To ensure the model learns to generalize across different camera angles and rink perspectives,
we use aggressive **Perspective Data Augmentations** (via Albumentations) during training.

## Setup steps
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough)
2. Upload the zip from your local machine: `python3 prepare_colab_upload.py` → uploads `colab_training_data.zip`
3. Run all cells
4. Download the output model and replace `src/calibration/board_segmentation_model.pth`

## 1 — Upload training data

In [ ]:
import sys
import os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    print('Upload colab_training_data.zip')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_name}')
else:
    print('Running locally. No need to upload zip.')


In [ ]:
import zipfile
if IN_COLAB:
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('.')
else:
    print('Running locally. Using local annotation_frames directory.')


## 2 — Install dependencies

In [ ]:
!pip install -q opencv-python-headless scipy albumentations


## 3 — Model definition

In [ ]:
import torch
import torch.nn as nn

TARGET_WIDTH  = 640
TARGET_HEIGHT = 360

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self._conv_block(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self._conv_block(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = self._conv_block(128, 256)
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self._conv_block(256, 128)
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(128, 64)
        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(64, 32)
        self.final = nn.Conv2d(32, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.upconv3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))
        return self.sigmoid(self.final(d1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
model = UNet().to(device)

# Load pre-trained weights if present
import os
if os.path.exists('board_segmentation_model.pth'):
    model.load_state_dict(torch.load('board_segmentation_model.pth', map_location=device))
    print('Loaded pre-trained weights — fine-tuning')
else:
    print('Training from scratch')

## 4 — Ground-truth mask generation from annotations

In [ ]:
import cv2
import numpy as np
import glob

def extract_mask_from_annotation(ann_bgr, orig_bgr=None, is_mask=False):
    if is_mask:
        if len(ann_bgr.shape) == 3:
            ann_bgr = cv2.cvtColor(ann_bgr, cv2.COLOR_BGR2GRAY)
        return (ann_bgr > 0).astype(np.uint8) * 255
    else:
        if orig_bgr is not None:
            diff = cv2.absdiff(ann_bgr, orig_bgr)
            diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
            mask = (diff_gray > 20).astype(np.uint8) * 255
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)))
            return mask
        else:
            hsv = cv2.cvtColor(ann_bgr, cv2.COLOR_BGR2HSV)
            green_mask = cv2.inRange(hsv, np.array([35, 50, 50]), np.array([85, 255, 255]))
            green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
            return green_mask

ANN_DIR = 'annotation_frames/new_batch/annotated'
PSEUDO_DIR = 'annotation_frames/pseudo_labeled'
PSEUDO_DIR = 'annotation_frames/pseudo_labeled'
pairs = []
processed_bases = set()

mask_files = glob.glob(f'{ANN_DIR}/*-mask.png') + glob.glob(f'{PSEUDO_DIR}/*-mask.png')
annotated_files = glob.glob(f'{ANN_DIR}/*-annotated.png') + glob.glob(f'{ANN_DIR}/*-annotate.png') + glob.glob(f'{PSEUDO_DIR}/*-annotated.png')

for m_path in mask_files:
    base = m_path.replace('-mask.png', '')
    orig_path = f'{base}.jpg'
    orig = cv2.imread(orig_path)
    mask_img = cv2.imread(m_path, cv2.IMREAD_GRAYSCALE)
    if orig is None or mask_img is None: continue
    mask = extract_mask_from_annotation(mask_img, is_mask=True)
    board_ratio = (mask > 0).mean()
    if board_ratio > 0.99:
        print(f"  SKIP {base.split('/')[-1]}: {board_ratio:.1%} board (too much)")
        continue
    board_ratio = (mask > 0).mean()
    if board_ratio > 0.99:
        print(f"  SKIP {base.split('/')[-1]}: {board_ratio:.1%} board (too much)")
        continue
    board_ratio = (mask > 0).mean()
    if board_ratio > 0.99:
        print(f"  SKIP {base.split('/')[-1]}: {board_ratio:.1%} board (too much)")
        continue
    pairs.append((orig, mask))
    processed_bases.add(base)
    print(f'  OK {base.split("/")[-1]} (-mask): {(mask>0).mean():.1%} board')

for a_path in annotated_files:
    base = a_path.replace('-annotated.png', '').replace('-annotate.png', '')
    if base in processed_bases: continue
    orig_path = f'{base}.jpg'
    orig = cv2.imread(orig_path)
    ann = cv2.imread(a_path)
    if orig is None or ann is None: continue
    mask = extract_mask_from_annotation(ann, orig_bgr=orig, is_mask=False)
    board_ratio = (mask > 0).mean()
    if board_ratio > 0.99:
        print(f"  SKIP {base.split('/')[-1]}: {board_ratio:.1%} board (too much)")
        continue
    board_ratio = (mask > 0).mean()
    if board_ratio > 0.99:
        print(f"  SKIP {base.split('/')[-1]}: {board_ratio:.1%} board (too much)")
        continue
    board_ratio = (mask > 0).mean()
    if board_ratio > 0.99:
        print(f"  SKIP {base.split('/')[-1]}: {board_ratio:.1%} board (too much)")
        continue
    pairs.append((orig, mask))
    processed_bases.add(base)
    print(f'  OK {base.split("/")[-1]} (-annotated): {(mask>0).mean():.1%} board')

print(f'\n{len(pairs)} frames loaded')


## 5 — Visualize ground truth (sanity check)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (orig, mask) in zip(axes.flat, pairs):
    vis = orig.copy()
    vis[mask > 0] = (vis[mask > 0] * 0.4 + np.array([0, 200, 0]) * 0.6).astype(np.uint8)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Board: {(mask>0).mean():.1%}')
    ax.axis('off')
plt.suptitle('Ground Truth Masks (GREEN = board zone)', fontsize=14)
plt.tight_layout()
plt.show()

## 6 — Dataset & Training

In [ ]:
# PRE-RESIZE ALL IMAGES IN MEMORY ONCE
TARGET_WIDTH  = 640
TARGET_HEIGHT = 360

print(f"Resizing {len(pairs)} frames in RAM to 640x360...")
resized_pairs = []
for orig, mask in pairs:
    orig_resized = cv2.resize(orig, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
    mask_resized = cv2.resize(mask, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_NEAREST)
    resized_pairs.append((orig_resized, mask_resized))

pairs = resized_pairs
print("✅ Done! All images are now pre-resized in RAM.")

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
import time

model_logits.train()
dl_times = []
fw_times = []
bw_times = []

print("Profiling first 10 batches...")
t0 = time.time()
for i, (x_b, y_b) in enumerate(dataloader):
    t_dl = time.time() - t0
    dl_times.append(t_dl)
    
    t_start_fw = time.time()
    x_b, y_b = x_b.to(device), y_b.to(device)
    out = model_logits(x_b)
    loss = criterion(out, y_b)
    t_fw = time.time() - t_start_fw
    fw_times.append(t_fw)
    
    t_start_bw = time.time()
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_logits.parameters(), 1.0)
    optimizer.step()
    t_bw = time.time() - t_start_bw
    bw_times.append(t_bw)
    
    if i >= 10:  # Profile 10 batches
        break
    t0 = time.time()

print("\n--- PROFILING RESULTS ---")
print(f"Average Dataloader (CPU Augmentation) time per batch: {sum(dl_times)/len(dl_times):.4f}s")
print(f"Average Forward pass (GPU) time per batch: {sum(fw_times)/len(fw_times):.4f}s")
print(f"Average Backward pass (GPU) time per batch: {sum(bw_times)/len(bw_times):.4f}s")
print(f"Total projected time per epoch (212 batches): {(sum(dl_times)+sum(fw_times)+sum(bw_times))/len(dl_times) * 212:.2f}s")

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
import torch.nn as nn

EPOCHS         = 150
LR             = 5e-4
AUGMENT_FACTOR = 5  # User optimized multiplier
BATCH_SIZE     = 8

# Ensure GPU/CPU check is explicit
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active training device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Aggressive perspective and photometric augmentations (Pre-resized, so no A.Resize is needed!)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    # Crucial for Option B: simulates different camera angles across the ice!
    A.Perspective(scale=(0.05, 0.15), keep_size=True, p=0.8),
    A.Affine(scale=(0.8, 1.2), translate_percent=(-0.1, 0.1), rotate=(-5, 5), p=0.7),
    A.GridDistortion(p=0.4), # Added advanced distortion for lens curvature
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), p=0.5),
    A.GaussNoise(std_range=(0.1, 0.3), p=0.5),
    # Simulate players/refs blocking the boards
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(20, 80), hole_width_range=(20, 80), p=0.6),
    A.Normalize(mean=(0,0,0), std=(1,1,1)), # Keep 0-1 scale like previous logic
    ToTensorV2()
])

class BoardDataset(Dataset):
    def __init__(self, pairs, augment=True, augment_factor=5):
        self.augment = augment
        self.augment_factor = augment_factor
        self.resized_pairs = []
        
        print(f"Pre-resizing {len(pairs)} frames to {TARGET_WIDTH}x{TARGET_HEIGHT} to optimize training speed...")
        for orig, mask in pairs:
            # Convert BGR to RGB once to save time in __getitem__
            img_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
            
            # Mask resizing - using NEAREST to keep binary values intact
            mask_resized = cv2.resize(mask, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_NEAREST)
            
            self.resized_pairs.append((img_resized, mask_resized))
        print("Pre-resizing completed.")
        
    def __len__(self):
        if self.augment:
            return len(self.resized_pairs) * self.augment_factor
        return len(self.resized_pairs)

    def __getitem__(self, idx):
        real_idx = idx % len(self.resized_pairs)
        img, mask = self.resized_pairs[real_idx]
        
        # First copy of each frame is kept clean/unaugmented
        is_base = (idx < len(self.resized_pairs))
        
        if self.augment and not is_base:
            # Albumentations transforms on pre-resized 640x360 image
            augmented = train_transform(image=img, mask=mask)
            x = augmented['image']
            y = augmented['mask']
        else:
            # Unaugmented baseline: Convert to tensor and normalize to [0, 1]
            x = torch.from_numpy(img).float().permute(2, 0, 1) / 255.0
            y = torch.from_numpy(mask).float()
            
        y = (y > 0).float().unsqueeze(0)
        return x, y

dataset    = BoardDataset(pairs, augment=True, augment_factor=AUGMENT_FACTOR)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)  # num_workers=0 to avoid Colab process overhead
print(f'Dataset: {len(dataset)} samples ({len(pairs)} frames × {AUGMENT_FACTOR})')

# Logits model for training (skip sigmoid for BCEWithLogitsLoss)
class UNetLogits(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x):
        e1 = self.base.enc1(x)
        e2 = self.base.enc2(self.base.pool1(e1))
        e3 = self.base.enc3(self.base.pool2(e2))
        b  = self.base.bottleneck(self.base.pool3(e3))
        
        # Safe upsampling (matching ml_board_detector.py robust logic)
        import torch.nn.functional as F
        up3 = self.base.upconv3(b)
        if up3.shape != e3.shape: up3 = F.interpolate(up3, size=e3.shape[2:])
        d3 = self.base.dec3(torch.cat([up3, e3], dim=1))
        
        up2 = self.base.upconv2(d3)
        if up2.shape != e2.shape: up2 = F.interpolate(up2, size=e2.shape[2:])
        d2 = self.base.dec2(torch.cat([up2, e2], dim=1))
        
        up1 = self.base.upconv1(d2)
        if up1.shape != e1.shape: up1 = F.interpolate(up1, size=e1.shape[2:])
        d1 = self.base.dec1(torch.cat([up1, e1], dim=1))
        
        return self.base.final(d1)

model_logits = UNetLogits(model).to(device)
pos_weight   = torch.tensor([3.0]).to(device)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, inputs, targets, smooth=1):
        inputs_sig = torch.sigmoid(inputs)
        inputs_flat = inputs_sig.view(-1)
        targets_flat = targets.view(-1)
        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs_flat.sum() + targets_flat.sum() + smooth)
        bce_loss = self.bce(inputs, targets)
        return bce_loss + dice_loss

criterion    = DiceBCELoss(pos_weight=pos_weight)
optimizer    = optim.AdamW(model_logits.parameters(), lr=LR, weight_decay=1e-4)
scheduler    = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.02)

losses = []
best_loss = float('inf')

import time
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model_logits.train()
    ep_loss = 0.0
    for x_b, y_b in dataloader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model_logits(x_b), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_logits.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    avg = ep_loss / len(dataloader)
    losses.append(avg)
    elapsed = time.time() - start_time
    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), 'board_segmentation_model.pth')
    print(f'Epoch {epoch:3d}/{EPOCHS} | loss={avg:.4f} | best={best_loss:.4f} | time={elapsed:.2f}s')

print(f'\nTraining complete. Best loss: {best_loss:.4f}')


## 7 — Loss curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('Training Loss'); plt.grid(True)
plt.tight_layout(); plt.show()

## 8 — Validate predictions vs ground truth

In [ ]:
model.eval()
import numpy as np

# Compute IoU over ALL pairs without plotting them to avoid crashing Colab
print(f"Calculating IoU over all {len(pairs)} frames...")
ious = []
for orig, gt_mask in pairs:
    h, w = orig.shape[:2]
    # Check if image is already TARGET size or needs resizing
    if (w, h) != (TARGET_WIDTH, TARGET_HEIGHT):
        img = cv2.resize(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB),
                          (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
    else:
        img = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
        
    x = torch.from_numpy(img).float().permute(2,0,1).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        prob = model(x).squeeze().cpu().numpy()
        
    prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    pred_mask = (prob > 0.18).astype(np.uint8) * 255

    gt_b   = gt_mask   > 0
    pred_b = pred_mask > 0
    inter  = (gt_b & pred_b).sum()
    union  = (gt_b | pred_b).sum()
    iou    = inter / union if union > 0 else 0.0
    ious.append(iou)

mean_iou = np.mean(ious)
print(f"Mean IoU across all validation frames: {mean_iou:.2%}")

# Select a small sample of 6 frames to visualize safely
num_vis = min(6, len(pairs))
indices = np.linspace(0, len(pairs) - 1, num_vis, dtype=int)
print(f"Visualizing {num_vis} sample frames...")

fig, axes = plt.subplots(num_vis, 3, figsize=(18, 4 * num_vis))
if num_vis == 1: axes = [axes]

for plot_idx, idx in enumerate(indices):
    orig, gt_mask = pairs[idx]
    h, w = orig.shape[:2]
    
    if (w, h) != (TARGET_WIDTH, TARGET_HEIGHT):
        img = cv2.resize(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB),
                          (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_AREA)
    else:
        img = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
        
    x = torch.from_numpy(img).float().permute(2,0,1).unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        prob = model(x).squeeze().cpu().numpy()
        
    prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    pred_mask = (prob > 0.18).astype(np.uint8) * 255
    iou = ious[idx]

    orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)

    def overlay(base, mask, color):
        vis = base.copy()
        vis[mask > 0] = (vis[mask > 0] * 0.4 + np.array(color) * 0.6).astype(np.uint8)
        return vis

    axes[plot_idx][0].imshow(orig_rgb); axes[plot_idx][0].set_title('Original'); axes[plot_idx][0].axis('off')
    axes[plot_idx][1].imshow(overlay(orig_rgb, gt_mask,   [0, 200, 0]))
    axes[plot_idx][1].set_title('Ground Truth'); axes[plot_idx][1].axis('off')
    axes[plot_idx][2].imshow(overlay(orig_rgb, pred_mask, [0, 120, 255]))
    axes[plot_idx][2].set_title(f'Predicted  IoU={iou:.2%}'); axes[plot_idx][2].axis('off')

plt.suptitle(f'GT (green) vs Predicted (blue) - Mean IoU: {mean_iou:.2%}', fontsize=14)
plt.tight_layout(); plt.show()


## 9 — Download the trained model

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download('board_segmentation_model.pth')
    print('Downloaded! Replace src/calibration/board_segmentation_model.pth in your repo.')
else:
    import shutil
    shutil.copy('board_segmentation_model.pth', 'src/calibration/board_segmentation_model.pth')
    print('Saved model locally to src/calibration/board_segmentation_model.pth')
